In [2]:
# Prevent tokenizer fork + DataLoader worker crashes (set before any other imports)
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
import mteb
from torch.utils.data import DataLoader
from datasets import Dataset

# Load by model name (MTEB uses the registered meta + NemotronColEmbedVL loader)
MODEL_NAME = "nvidia/llama-nemotron-colembed-vl-3b-v2"
model = mteb.get_model(MODEL_NAME, device="cuda")

# ColEmbed's HF model hardcodes num_workers=8 in forward_documents -> OOM "Killed". Patch to num_workers=0.
def _patched_forward_documents(self, corpus, batch_size=8):
    images, texts = [], []
    for doc in corpus:
        text = doc.get("text", "")
        image = doc.get("image")
        if image is not None and image.mode != "RGB":
            image = image.convert("RGB")
        images.append(image)
        texts.append(text)
    dataset = Dataset.from_dict({"image": images, "text": texts})
    dataloader = DataLoader(dataset=dataset, batch_size=batch_size, collate_fn=self.process_documents,
                           shuffle=False, num_workers=0, pin_memory=False, drop_last=False)
    return self._extract_embeddings(dataloader=dataloader, is_query=False)
model.model.forward_documents = _patched_forward_documents.__get__(model.model, type(model.model))


/home/fidaa/Documents/MILA/IFT6289/project/ift6289_rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some kwargs in processor config are unused and will not have any effect: num_image_token, use_thumbnail, template, q_max_length, query_prefix, pad_to_multiple_of, p_max_length, image_size, norm_type, padding, num_channels, dynamic_image_size, max_input_tiles, passage_prefix, system_message. 
Loading checkpoint shards: 100%|██████████| 2/2 [00:17<00:00,  8.57s/it]


In [ ]:
# MTEB() only accepts `tasks` (list of task instances) and optional `err_logs_path`.
# Use get_tasks() to get task instances and filter by language.
tasks = mteb.get_tasks(tasks=["Vidore3PhysicsRetrieval"], languages=["eng-Latn"])
evaluation = mteb.evaluate(model, tasks=tasks, num_proc=0, encode_kwargs={"batch_size": 16})


Evaluating task Vidore3PhysicsRetrieval:   0%|          | 0/1 [05:36<?, ?it/s]


RuntimeError: DataLoader worker (pid 10841) is killed by signal: Killed. 

: 